# RFM Analysis:-
RFM Analysis is a customer segmentation technique used in marketing and business analytics to identify the most valuable customers.
Recency (R) - How recently a customer made a purchase.;LOWER days = BETTER
Frequency (F) - How often a customer purchases.; HIGHER orders = BETTER
Monetary (M) - How much money they spend.; HIGHER spend = BETTER

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('ecommerce_orders.csv')

In [3]:
df.describe()

,quantity,unit_price,total_amount
count,3043.000000,3043.000000,3043.000000
mean,2.501807,17009.308945,41809.965919
std,1.122471,34777.257688,94351.758530
min,1.000000,214.480000,274.110000
25%,1.500000,1817.445000,3706.330000
50%,2.000000,3837.530000,8678.420000
75%,4.000000,9228.285000,23887.565000
max,4.000000,179512.430000,718049.720000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     3043 non-null   object 
 1   order_id        3043 non-null   object 
 2   order_date      3043 non-null   object 
 3   product_name    3043 non-null   object 
 4   category        3043 non-null   object 
 5   quantity        3043 non-null   int64  
 6   unit_price      3043 non-null   float64
 7   total_amount    3043 non-null   float64
 8   payment_method  3043 non-null   object 
 9   city            3043 non-null   object 
 10  returned        3043 non-null   bool   
 11  true_segment    3043 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(8)
memory usage: 264.6+ KB


changing datatype of order_date where in the dataset its stored in object

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'],format='%d-%m-%Y')
# pd.to_datetime converts text like "2023-05-12" into a date format.

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customer_id     3043 non-null   object        
 1   order_id        3043 non-null   object        
 2   order_date      3043 non-null   datetime64[ns]
 3   product_name    3043 non-null   object        
 4   category        3043 non-null   object        
 5   quantity        3043 non-null   int64         
 6   unit_price      3043 non-null   float64       
 7   total_amount    3043 non-null   float64       
 8   payment_method  3043 non-null   object        
 9   city            3043 non-null   object        
 10  returned        3043 non-null   bool          
 11  true_segment    3043 non-null   object        
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 264.6+ KB


In [10]:
df.head(10)

,customer_id,order_id,order_date,product_name,category,quantity,unit_price,total_amount,payment_method,city,returned,true_segment
0,C0001,ORD00001,2023-12-24,Laptop Stand,Electronics,1,5945.88,5945.88,UPI,Delhi,False,champion
1,C0001,ORD00002,2023-12-22,Samsung Galaxy S23,Electronics,1,88745.52,88745.52,Net Banking,Delhi,False,champion
2,C0001,ORD00003,2023-11-23,Wireless Earbuds,Electronics,3,8509.24,25527.73,UPI,Delhi,False,champion
3,C0001,ORD00004,2023-12-06,Perfume,Beauty,3,5688.05,17064.16,Debit Card,Delhi,False,champion
4,C0001,ORD00005,2023-12-09,Samsung Galaxy S23,Electronics,3,95197.09,285591.28,Net Banking,Delhi,False,champion
5,C0001,ORD00006,2023-11-05,Bed Sheet Set,Home,4,1999.87,7999.49,COD,Delhi,False,champion
6,C0001,ORD00007,2023-12-06,Data Science Book,Books,3,1398.40,4195.21,UPI,Delhi,False,champion
7,C0001,ORD00008,2023-12-12,Moisturizer,Beauty,3,1115.33,3345.98,Credit Card,Delhi,False,champion
8,C0001,ORD00009,2023-11-24,Formal Shirt,Fashion,3,2856.30,8568.90,Debit Card,Delhi,False,champion
9,C0001,ORD00010,2023-10-16,Moisturizer,Beauty,1,1220.84,1220.84,Credit Card,Delhi,False,champion


In [ ]:
df = df[df['returned'] == False]
# just removing the returned orders, as they are not relevant for the RFM analysis.

In [18]:
reference_date = df['order_date'].max() + pd.Timedelta(days=1)
print(f"Reference date: {reference_date.date()}")
#Timedelta in pandas represents a duration of time
# max() gives us the latest order date, and we add one day to set the reference date for our analysis.
# max()+1


Reference date: 2024-01-01


as we know based on dataset we have order_date till 31st dec 2023;+1 means it is 1st jan 2024

In [ ]:
rfm = df.groupby('customer_id').agg(
recency  = ('order_date',   lambda x: (reference_date - x.max()).days),
# this checkes the customer id and there orderdates using that taking orderdate as x 
# and it makes list of orderdate of each customer_id and then picks the max orderdate out of it and then
# subtracts it from reference date and gives us the recency in days. 
frequency = ('order_id',    'nunique'),
# this checks the customer id and there order_id this counts the number of unique order_id for each customer_id and gives us the frequency.
monetary  = ('total_amount', 'sum') ).reset_index()
# this checks the customer id and there total_amount this sums up the total_amount for each customer_id and gives us the monetary value.
print("\n📊 Raw RFM sample:")
print(rfm.head(5))
print("\n RFM Statistics:")
print(rfm.describe().round(1))


📊 Raw RFM sample:
  customer_id  recency  frequency    monetary
0       C0001        8         14   516680.28
1       C0002       50          5    65651.11
2       C0003      140          3   203185.46
3       C0004        3         17  1193414.12
4       C0005       67          4   187150.09

 RFM Statistics:
       recency  frequency   monetary
count    496.0      496.0      496.0
mean      89.7        5.8   245039.0
std       99.8        5.1   389499.2
min        1.0        1.0      406.9
25%       20.0        2.0    13901.1
50%       41.0        5.0    65065.7
75%      127.5        7.0   320012.2
max      365.0       22.0  2824263.7


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2894 entries, 0 to 3042
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customer_id     2894 non-null   object        
 1   order_id        2894 non-null   object        
 2   order_date      2894 non-null   datetime64[ns]
 3   product_name    2894 non-null   object        
 4   category        2894 non-null   object        
 5   quantity        2894 non-null   int64         
 6   unit_price      2894 non-null   float64       
 7   total_amount    2894 non-null   float64       
 8   payment_method  2894 non-null   object        
 9   city            2894 non-null   object        
 10  returned        2894 non-null   bool          
 11  true_segment    2894 non-null   object        
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 274.1+ KB


# STEP B: ASSIGN SCORES 1–5 TO EACH DIMENSION

In [29]:
# RECENCY SCORE: note labels=[5,4,3,2,1] (REVERSED!)
# BEST = score 5
# pd.qcut() is used because we want to rank customers relative to other customers.For example 1000 values for 10 quantiles
rfm['R'] = pd.qcut(rfm['recency'],q=5,labels=[5, 4, 3, 2, 1]).astype(int) # reversed: best recent = 5
# q=5 means we want to divide the recency values into 5 equal groups
# astype(int) converts the category labels to actual integers
# example:- if recency values are 20,30,5,60 then the rank will be for 5=5, for 20=4, for 30=3 and for 60=1 because 5 is the most recent and 60 is the least recent.
# most recent customers receive the highest R score of 5.
# FREQUENCY SCORE:more orders = higher score
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'),q=5,labels=[1, 2, 3, 4, 5]).astype(int)
# rank(method='first') assigns ranks to first occurrence of each frequency value, ensuring that ties are ranked in the order they appear in the data.
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),q=5,labels=[1,2,3,4,5]).astype(int)
# higher monetary value = higher score
# COMBINED RFM SCORE (3–15 range)
rfm['RFM_score'] = rfm['R'] + rfm['F'] + rfm['M']
# Simple addition. Score 15 = perfect customer (5+5+5)
print("\n RFM scores calculated!")



 RFM scores calculated!


In [32]:
print(rfm[['customer_id','recency','R','frequency','F','monetary','M','RFM_score']].head(10))

  customer_id  recency  R  frequency  F    monetary  M  RFM_score
0       C0001        8  5         14  5   516680.28  5         15
1       C0002       50  3          5  3    65651.11  3          9
2       C0003      140  2          3  2   203185.46  4          8
3       C0004        3  5         17  5  1193414.12  5         15
4       C0005       67  2          4  3   187150.09  4          9
5       C0006       15  5          5  3    18089.80  2         10
6       C0007       36  3          5  3   242815.09  4         10
7       C0008       40  3          2  1   180167.39  4          8
8       C0009       39  3          4  3    42777.06  3          9
9       C0010       42  3          7  4   159535.35  4         11


# if the RFM_Score is 15 - good customer; 3 - bad customer.

# STEP C: LABEL CUSTOMER SEGMENTS

In [38]:
def label_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'           # Recent + Frequent + Big spender
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customer'     # Consistently good
    elif r >= 4 and f <= 2:
        return 'New Customer'       # Just started buying
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'            # Good customer gone quiet — DANGER!
    elif r <= 2 and f <= 2:
        return 'Lost'               # Haven't seen them in months
    else:
        return 'Potential Loyalist' # Middle of the road — can be nurtured
rfm['segment'] = rfm.apply(label_segment, axis=1)

In [41]:
print(rfm[['customer_id','recency','R','frequency','F','monetary','M','RFM_score','segment']].head(10))

  customer_id  recency  R  frequency  F    monetary  M  RFM_score  \
0       C0001        8  5         14  5   516680.28  5         15   
1       C0002       50  3          5  3    65651.11  3          9   
2       C0003      140  2          3  2   203185.46  4          8   
3       C0004        3  5         17  5  1193414.12  5         15   
4       C0005       67  2          4  3   187150.09  4          9   
5       C0006       15  5          5  3    18089.80  2         10   
6       C0007       36  3          5  3   242815.09  4         10   
7       C0008       40  3          2  1   180167.39  4          8   
8       C0009       39  3          4  3    42777.06  3          9   
9       C0010       42  3          7  4   159535.35  4         11   

              segment  
0            Champion  
1      Loyal Customer  
2                Lost  
3            Champion  
4             At Risk  
5  Potential Loyalist  
6      Loyal Customer  
7  Potential Loyalist  
8      Loyal Customer  


In [40]:
# ── SEGMENT SUMMARY ──
print("\n Customer Segment Breakdown:")
summary = rfm.groupby('segment').agg(
customers = ('customer_id', 'count'),
avg_recency  = ('recency',   'mean'),
avg_frequency= ('frequency', 'mean'),
avg_monetary = ('monetary',  'mean'),
total_revenue= ('monetary',  'sum')
).round(1)
summary['revenue_share_pct'] = (summary['total_revenue'] /
                                 summary['total_revenue'].sum() * 100).round(1)
print(summary)


 Customer Segment Breakdown:
                    customers  avg_recency  avg_frequency  avg_monetary  \
segment                                                                   
At Risk                    53        105.8            5.1      176518.4   
Champion                  105         15.2           14.0      796128.5   
Lost                      119        242.0            1.7       34337.8   
Loyal Customer             90         37.0            6.3      214588.2   
New Customer               58         16.6            1.9       41922.5   
Potential Loyalist         71         59.1            3.9       38869.5   

                    total_revenue  revenue_share_pct  
segment                                               
At Risk                 9355474.3                7.7  
Champion               83593489.6               68.8  
Lost                    4086201.0                3.4  
Loyal Customer         19312941.5               15.9  
New Customer            2431506.7      

In [42]:
rfm.to_csv('rfm_segments.csv', index=False)